## Ejercicio 1: Alerta epidemiológica
Muestra de LCR de un paciente pediátrico con rigidez de nuca, fiebre alta y petequias. Se amplificó una región del 16S rDNA y se identifica la bacteria mediante BLAST (nucleótido contra nucleótido) contra la base de datos `16S_ribosomal_RNA`, de forma **local** y **online**.

In [10]:
from Bio.Blast import NCBIXML
from Bio.Seq import Seq
import os
import subprocess
import tarfile

def resumen(xml_path, n=10):
    with open(xml_path) as f:
        rec = NCBIXML.read(f)
    print(f"Consulta: {rec.query_letters} nt | BD: {rec.database}")
    print(f"{'#':<3}{'Especie':<55}{'Ident.':>8}{'Cover':>8}{'E-value':>11}")
    for i, aln in enumerate(rec.alignments[:n], 1):
        h = aln.hsps[0]
        print(f"{i:<3}{aln.hit_def[:53]:<55}"
              f"{h.identities/h.align_length*100:>7.1f}%"
              f"{h.align_length/rec.query_letters*100:>7.0f}%{h.expect:>11.1e}")
    return rec

secuencia_paciente_lcr = " GACGGCAGCACAGAGAAGCTTGCTTCTCGGGTGGCGAGTGGCGAACGGGTGAGTAA"

secuencia = secuencia_paciente_lcr.strip().upper()
assert set(secuencia) <= set("ACGT"), "La cadena contiene caracteres no válidos"
print(f"Longitud: {len(secuencia)} nt")
print(secuencia)

Longitud: 56 nt
GACGGCAGCACAGAGAAGCTTGCTTCTCGGGTGGCGAGTGGCGAACGGGTGAGTAA


### 1.1 BLAST local (BLAST+ y base de datos `16S_ribosomal_RNA`)
La consulta es muy corta (56 nt), así que se usa `-task blastn` (word size 11) en lugar de megablast, que con word size 28 puede dar pocos resultados.

In [11]:
DB_ARCHIVE = "blastdb/16S_ribosomal_RNA.tar.gz"
DB_LOCAL = "blastdb/16S_ribosomal_RNA"
if not os.path.exists(DB_LOCAL + ".nsq"):
    if not os.path.isfile(DB_ARCHIVE):
        raise FileNotFoundError(f"No se encuentra la base de datos: {DB_ARCHIVE}")
    with tarfile.open(DB_ARCHIVE, "r:gz") as archive:
        archive.extractall("blastdb")


with open("query_lcr.fasta", "w") as f:
    f.write(">paciente_lcr\n" + secuencia + "\n")

cmd = ["blastn", "-task", "blastn",
       "-query", "query_lcr.fasta",
       "-db", DB_LOCAL,
       "-evalue", "0.05",
       "-outfmt", "5",
       "-out", "resultado_local.xml"]

res = subprocess.run(cmd, capture_output=True, text=True)
if res.returncode != 0:
    print("blastn falló (código", res.returncode, "):\n", res.stderr)
else:
    resumen("resultado_local.xml")

blastn falló (código 2 ):
 BLAST Database error: LMDB runtime error: mdb_env_open: El sistema no puede encontrar la ruta especificada.





### 1.2 BLAST online
**Nota:** la petición con `NCBIWWW.qblast` quedó más de una hora en la cola de NCBI sin devolver resultado. Por la lentitud del servidor (y con el visto bueno del profesor), el BLAST online se ejecutó desde la web de NCBI (<https://blast.ncbi.nlm.nih.gov>): *Nucleotide BLAST*, base de datos *16S ribosomal RNA sequences (Bacteria and Archaea)*, programa *Somewhat similar sequences (blastn)*. El resultado se descargó en formato **XML** y se lee aquí con `NCBIXML`.

In [12]:
rec_online = resumen("GN93KTEF016-Alignment.xml")

Consulta: 56 nt | BD: rRNA_typestrains/16S_ribosomal_RNA
#  Especie                                                  Ident.   Cover    E-value
1  Neisseria meningitidis strain M1027 16S ribosomal RNA    100.0%    100%    2.1e-22
2  Neisseria gonorrhoeae strain NCTC 8375 16S ribosomal      98.2%    100%    9.0e-21
3  Neisseria sicca ATCC 29256 16S ribosomal RNA, partial     98.2%    100%    9.0e-21
4  Neisseria cinerea strain ATCC 14685 16S ribosomal RNA     98.2%    100%    9.0e-21
5  Neisseria mucosa strain DSM 17611 16S ribosomal RNA,      98.2%    100%    9.0e-21
6  Neisseria macacae strain M-740 16S ribosomal RNA, par     98.2%    100%    9.0e-21
7  Neisseria mucosa strain N16 16S ribosomal RNA, partia     98.2%    100%    9.0e-21
8  Neisseria perflava strain Branham 7078 16S ribosomal      98.2%    100%    9.0e-21
9  Neisseria weaveri strain 8142 16S ribosomal RNA, part     98.2%    100%    9.0e-21
10 Neisseria dumasiana strain 93087 16S ribosomal RNA, p     98.2%    100%    9.0e-

### 1.3 Conclusión
- El mejor hit es **_Neisseria meningitidis_** (cepa M1027): 56/56 nt idénticos (100 %), cobertura 100 %, sin gaps, E-value ≈ 2·10⁻²².
- El resto de hits son otras especies de *Neisseria* con 55/56 (98,2 %), p. ej. *N. gonorrhoeae* (difiere en la posición 14) y las especies comensales (difieren en la 28). Con solo 56 nt el 16S está demasiado conservado en el género para descartar con rigor a las especies vecinas; los hits de *Vibrio*, *Pantoea*, *Lysobacter*… tienen 4-6 mismatches y son ruido de una consulta corta.
- Resultado local y online coinciden en la identificación (*N. meningitidis*) (comprobar la salida de la celda local).
- Contexto clínico (meningitis con petequias en un paciente pediátrico + *N. meningitidis* en LCR): **meningitis meningocócica, sí es una emergencia vital**. Requiere aislamiento, antibioterapia inmediata y profilaxis de contactos. Como matiz, conviene confirmar con una región más larga del 16S o con una PCR específica.

---
## Ejercicio 2: La caza de enzimas industriales
ADN crudo de una bacteria de los géiseres de Yellowstone, sin saber **en qué cadena ni en qué marco de lectura** está el gen. Eso significa que hay que considerar los **6 marcos de lectura** (3 en la cadena directa y 3 en la complementaria inversa), traducir cada uno y buscar cuál codifica una proteína real.

In [13]:
adn_geiser = "CTCTCCTGGGACCTGGCCAAGGTGCGCACCGACCTGCCCCTGGAGGTGGACTTCGCCAAAAGGCGGGAGC"
seq = Seq(adn_geiser)
print(f"Longitud: {len(seq)} nt")

marcos = {}
for cadena, s in (("+", seq), ("-", seq.reverse_complement())):
    for f in range(3):
        sub = s[f:]
        sub = sub[: len(sub) // 3 * 3]
        marcos[f"{cadena}{f+1}"] = str(sub.translate())

for nombre, prot in marcos.items():
    print(f"Marco {nombre}: {prot}   (codones de parada: {prot.count('*')})")

Longitud: 70 nt
Marco +1: LSWDLAKVRTDLPLEVDFAKRRE   (codones de parada: 0)
Marco +2: SPGTWPRCAPTCPWRWTSPKGGS   (codones de parada: 0)
Marco +3: LLGPGQGAHRPAPGGGLRQKAG   (codones de parada: 0)
Marco -1: APAFWRSPPPGAGRCAPWPGPRR   (codones de parada: 0)
Marco -2: LPPFGEVHLQGQVGAHLGQVPGE   (codones de parada: 0)
Marco -3: SRLLAKSTSRGRSVRTLARSQE   (codones de parada: 0)


**Análisis de los 6 marcos:** ninguno contiene codones de parada (la secuencia es corta), así que no se descarta ninguno por ese criterio. Hay que usar la similitud con proteínas conocidas. El marco **+1** da un péptido de 23 aminoácidos con composición típica de proteína globular (L, W, D, K, V, R, T, E...), mientras que el resto de marcos son ricos en P, G, A, R, típico de traducciones en un marco incorrecto. Para confirmarlo, se lanza **`blastx`** (la consulta de nucleótidos se traduce en los 6 marcos y se compara contra una base de datos de proteínas), que resuelve a la vez la cadena y el marco.

In [14]:
xml_blastx = "blastx_geiser.xml"
if os.path.exists(xml_blastx):
    with open(xml_blastx) as f:
        rec = NCBIXML.read(f)
    for i, aln in enumerate(rec.alignments[:5], 1):
        h = aln.hsps[0]
        print(f"{i}. {aln.hit_def[:90]}")
        print(f"   Marco: {h.frame}  Ident.: {h.identities}/{h.align_length}  E-value: {h.expect:.1e}")
        print(f"   Query: {h.query}\n   Hit  : {h.sbjct}")
    print("\nProteína descubierta:", rec.alignments[0].hit_def)

1. DNA polymerase I [Bacillus subtilis]
   Marco: (1, 0)  Ident.: 23/23  E-value: 6.0e-05
   Query: LSWDLAKVRTDLPLEVDFAKRRE
   Hit  : LSWDLAKVRTDLPLEVDFAKRRE
2. PolI [Thermus sp. YSP2A.A1]
   Marco: (1, 0)  Ident.: 23/23  E-value: 6.1e-05
   Query: LSWDLAKVRTDLPLEVDFAKRRE
   Hit  : LSWDLAKVRTDLPLEVDFAKRRE
3. DNA polymerase I [Salmonella enterica] >gb|MHP1808696.1| DNA polymerase I [Salmonella ente
   Marco: (1, 0)  Ident.: 23/23  E-value: 6.1e-05
   Query: LSWDLAKVRTDLPLEVDFAKRRE
   Hit  : LSWDLAKVRTDLPLEVDFAKRRE
4. Taq polymerase 1 [eukaryotic synthetic construct] >dbj|BAJ61824.1| Taq polymerase 1 [eukar
   Marco: (1, 0)  Ident.: 23/23  E-value: 6.1e-05
   Query: LSWDLAKVRTDLPLEVDFAKRRE
   Hit  : LSWDLAKVRTDLPLEVDFAKRRE
5. DNA polymerase I, thermostable [Thermus aquaticus]
   Marco: (1, 0)  Ident.: 23/23  E-value: 6.2e-05
   Query: LSWDLAKVRTDLPLEVDFAKRRE
   Hit  : LSWDLAKVRTDLPLEVDFAKRRE

Proteína descubierta: DNA polymerase I [Bacillus subtilis]


### Ejercicio 3

### Ejercicio 4